# SLAB Steward — impact

What Steward has actually put into OpenStreetMap: how many changesets carry the
campaign hashtag, how many distinct people made them, and how many trail edits
they add up to.

```sh
cd tool/impact && uv run --with jupyter jupyter lab
```

The logic lives in `impact.py` next door and is imported below rather than
copied — a notebook that reimplements its own script drifts from it, and then
two answers to one question exist. `uv run tool/impact/impact.py --json` prints
the same numbers without a browser.

**Two things about the OSMCha API worth knowing before changing any query here**,
both found the hard way (docs/specs/analytics.md §5):

1. Its `hashtags` filter is **silently ignored** — the same failure mode as the
   OSM API's. The filter that works is `editor`, matching the `created_by` tag
   Steward stamps on every changeset; the hashtag is then re-checked client-side.
2. Query cost scales **steeply** with the date window. A floor of 2026-08-29
   answers in 0.4s, 2026-08-25 in 9s, and 2026-01-01 hangs past 150s. A hung
   cell here is far more likely a wide `SINCE` than a broken notebook.


## Setup


In [1]:
import importlib.util
from pathlib import Path

# Finds the repo root by walking up for pubspec.yaml, so this runs the same
# whether Jupyter was launched from tool/impact/ or from the repo root.
root = Path.cwd().resolve()
while not (root / 'pubspec.yaml').exists() and root != root.parent:
    root = root.parent

spec = importlib.util.spec_from_file_location(
    'impact', root / 'tool' / 'impact' / 'impact.py'
)
impact = importlib.util.module_from_spec(spec)
spec.loader.exec_module(impact)

token = impact.vault_get('OSMCHA_API_KEY')
assert token, (
    f'OSMCHA_API_KEY is missing or empty in {impact.VAULT}. Sign in at '
    'https://osmcha.org with your OSM account and copy the API token from '
    'your profile page.'
)
print(f'ready — filtering on editor={impact.EDITOR!r}, live since {impact.WENT_LIVE}')

ready — filtering on editor='SLAB Steward', live since 2026-08-29


## Parameters

`SINCE` defaults to the day `osmEnvironment` was flipped to `live` — the first
day any Steward edit could have reached OpenStreetMap, so there is nothing
earlier to find. Widening it will get slow long before it gets useful.


In [2]:
SINCE = impact.WENT_LIVE
HASHTAG = 'slabsteward'

## Fetch


In [3]:
raw = impact.fetch_changesets(token, SINCE)
changesets = [
    feature
    for feature in raw
    if impact.carries_hashtag(feature.get('properties', {}), HASHTAG)
]

# The gap between the two is the client-side hashtag re-check doing its job:
# OSMCha's `editor` filter is a substring match, so a future tool called
# 'SLAB Stewardship' would arrive in `raw` and be dropped here.
print(f'{len(raw)} from OSMCha, {len(changesets)} carrying #{HASHTAG}')

4 from OSMCha, 4 carrying #slabsteward


## The three numbers


In [4]:
summary = impact.summarise(changesets, HASHTAG)

for label, key in [
    ('changesets', 'changesets'),
    ('contributors', 'contributors'),
    ('trail edits', 'trail_edits'),
    ('first edit', 'first_edit'),
    ('last edit', 'last_edit'),
]:
    print(f'{label:<13} {summary[key]}')

changesets    4
contributors  1
trail edits   170
first edit    2026-08-30T04:32:16Z
last edit     2026-08-30T05:29:48Z


**"Trail edits", not "trails"** — and the distinction is worth keeping in any
sentence these numbers end up in. Steward writes one `<modify>` element per
trail, so this counts trail *writes*: a trail rated today and given a surface
next week counts twice. The distinct-trail figure would need one
`/changeset/{id}/download` per changeset against a courtesy-limited public API,
for a number strictly less useful when the question is how much the tool is
being used.


## Changeset by changeset


In [5]:
from html import escape

from IPython.display import HTML, display

rows = []
for feature in changesets:
    p = feature.get('properties', {})
    rows.append(
        '<tr>'
        f'<td><a href="https://www.openstreetmap.org/changeset/{feature["id"]}"'
        f' target="_blank">{feature["id"]}</a></td>'
        f'<td>{escape(str(p.get("date", ""))[:10])}</td>'
        f'<td>{escape(str(p.get("user", "")))}</td>'
        f'<td style="text-align:right">{impact.edit_count(p)}</td>'
        f'<td>{escape(str(p.get("comment", "")))}</td>'
        '</tr>'
    )

display(HTML(
    '<table><thead><tr>'
    '<th>changeset</th><th>date</th><th>mapper</th>'
    '<th>trail edits</th><th>comment</th>'
    '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'
))

changeset,date,mapper,trail edits,comment
188228523,2026-08-30,DannySlab,78,"Assert ebike not allowed policy for Duthie Hill bike park, per EMBA ebike page #slabsteward"
188227854,2026-08-30,DannySlab,72,Set difficulty and ebike tags for Soaring Eagle Park trails. #slabsteward
188227663,2026-08-30,DannySlab,2,Fill in missed tags on tennant trails. Complements Changeset 188227639 #slabsteward
188227639,2026-08-30,DannySlab,18,Set difficulty and ebike tags on Tennant Trailhead park. #slabsteward


## Who is using it


In [6]:
from collections import Counter

by_mapper = Counter()
edits_by_mapper = Counter()
for feature in changesets:
    p = feature.get('properties', {})
    name = p.get('user', 'unknown')
    by_mapper[name] += 1
    edits_by_mapper[name] += impact.edit_count(p)

for name, count in by_mapper.most_common():
    print(f'{name:<20} {count:>3} changesets  {edits_by_mapper[name]:>5} trail edits')

DannySlab              4 changesets    170 trail edits


---

## What these numbers are not

They are **not** the product funnel. How many people arrived, how many signed
in, how many gave up at the compliance gate — none of that is visible from
OpenStreetMap, which records only what succeeded. That is PostHog's job, and
the two must not be conflated: see docs/specs/analytics.md §5 and
docs/specs/analytics-dashboard.md.

They are, on the other hand, **complete**. Unlike the funnel, nothing here is
lost to ad blockers — OSM's record is the record.
